**IMPORTS**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle  # Para persistencia

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("Librerías de optimización y persistencia cargadas con éxito.")

Librerías de optimización y persistencia cargadas con éxito.


**CARGA Y CODIFICACIOND DE DATOS**

In [2]:
# Cargar datos procesados
ruta_csv = '../data/processed/bank_processed.csv'
df = pd.read_csv(ruta_csv)

# Convertir variables categóricas a numéricas (One-Hot Encoding)
categorical_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

X = df_encoded.drop(columns=['deposit'])
y = df_encoded['deposit']

# División estratificada (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Datos preparados para la fase de optimización.")

Datos preparados para la fase de optimización.


**EXPERIMENTO 1 - AJUSTE DE HIPERPARAMETROS RANDOM FOREST**

In [3]:
print("\nIniciando búsqueda de hiperparámetros para Random Forest (esto puede tardar unos segundos)...")

# Malla optimizada para garantizar el mejor rendimiento del Random Forest
param_grid = {
    'n_estimators': [150, 200],
    'max_depth': [12, 15],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

# Configurar la búsqueda cruzada buscando optimizar el F1-Score
grid_search_rf = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

best_rf = grid_search_rf.best_estimator_
print(f"Mejores parámetros encontrados: {grid_search_rf.best_params_}")

# Evaluar Random Forest Optimizado
y_pred_rf = best_rf.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
print(f"F1-Score del Random Forest Optimizado: {f1_rf:.4f}")


Iniciando búsqueda de hiperparámetros para Random Forest (esto puede tardar unos segundos)...
Mejores parámetros encontrados: {'max_depth': 15, 'min_samples_split': 5, 'n_estimators': 200}
F1-Score del Random Forest Optimizado: 0.8616


**EXPERIMENTO 2 - MODELO COMPARATIVO**

In [4]:
print("\nEntrenando modelo de comparación: Regresión Logística...")

# Inicializar y entrenar Regresión Logística (se aumenta max_iter para asegurar convergencia)
lr_model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)

# Evaluar Regresión Logística
y_pred_lr = lr_model.predict(X_test)
f1_lr = f1_score(y_test, y_pred_lr)
print(f"F1-Score de la Regresión Logística: {f1_lr:.4f}")


Entrenando modelo de comparación: Regresión Logística...
F1-Score de la Regresión Logística: 0.8206


**CUADRO COMPARATIVO DE RENDIMIENTO**

In [5]:
# Crear dataframe con los resultados para visualizar la tabla comparativa
resultados = pd.DataFrame({
    'Modelo': ['Random Forest Optimizado', 'Regresión Logística (Comparativo)'],
    'F1-Score (Test)': [f1_rf, f1_lr]
})

print("\n==================================================")
print("       --- TABLA COMPARATIVA DE RENDIMIENTO ---")
print("==================================================")
print(resultados.to_string(index=False))
print("==================================================\n")

# Decidir automáticamente cuál es el mejor modelo basado en el F1-Score
if f1_rf > f1_lr:
    best_model = best_rf
    nombre_mejor = "Random Forest"
    f1_ganador = f1_rf
else:
    best_model = lr_model
    nombre_mejor = "Regresión Logística"
    f1_ganador = f1_lr

print("🏆" * 25)
print(f"🥇 ¡EL MODELO GANADOR PARA PRODUCCIÓN ES: {nombre_mejor.upper()}!")
print(f"📈 Con un rendimiento de F1-Score de: {f1_ganador:.4f}")
print("🏆" * 25)


       --- TABLA COMPARATIVA DE RENDIMIENTO ---
                           Modelo  F1-Score (Test)
         Random Forest Optimizado         0.861635
Regresión Logística (Comparativo)         0.820634

🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
🥇 ¡EL MODELO GANADOR PARA PRODUCCIÓN ES: RANDOM FOREST!
📈 Con un rendimiento de F1-Score de: 0.8616
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆


**PERSISTENCIA DEL MODELO GANADOR**

In [6]:
# Definir la ruta de guardado dentro de la carpeta /models creada
ruta_modelo = '../models/modelo_bank_marketing.pkl'

# Guardar el modelo físicamente
with open(ruta_modelo, 'wb') as archivo:
    pickle.dump(best_model, archivo)

print(f"\n¡Persistencia completada con éxito!")
print(f"El archivo ejecutable del modelo quedó guardado de forma segura en: {ruta_modelo}")


¡Persistencia completada con éxito!
El archivo ejecutable del modelo quedó guardado de forma segura en: ../models/modelo_bank_marketing.pkl
